In [34]:
# import pandas
import pandas as pd
import numpy as np

In [35]:
# importint the data
df = pd.read_csv(r"C:\Users\user\Desktop\Study sess\dirty_cafe_sales.csv")

In [36]:
# inspecting data
df.head(20)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2,4,Credit Card,Takeaway,08/09/2023
1,TXN_4977031,Cake,4,3,12,Cash,In-store,16/05/2023
2,TXN_4271903,Cookie,4,1,ERROR,Credit Card,In-store,19/07/2023
3,TXN_7034554,Salad,2,5,10,UNKNOWN,UNKNOWN,27/04/2023
4,TXN_3160411,Coffee,2,2,4,Digital Wallet,In-store,11/06/2023
5,TXN_2602893,Smoothie,5,4,20,Credit Card,NaN,31/03/2023
6,TXN_4433211,UNKNOWN,3,3,9,ERROR,Takeaway,06/10/2023
7,TXN_6699534,Sandwich,4,4,16,Cash,UNKNOWN,28/10/2023
8,TXN_4717867,NaN,5,3,15,NaN,Takeaway,28/07/2023
9,TXN_2064365,Sandwich,5,4,20,NaN,In-store,31/12/2023


In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [38]:
df.shape

(10000, 8)

In [39]:
df.isnull().sum()

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

In [40]:
# check for duplicates
df.duplicated().sum()

np.int64(0)

In [41]:
# standardize the columns
df.columns = df.columns.str.strip()
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(" ","_")


In [42]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   transaction_id    10000 non-null  object
 1   item              9667 non-null   object
 2   quantity          9862 non-null   object
 3   price_per_unit    9821 non-null   object
 4   total_spent       9827 non-null   object
 5   payment_method    7421 non-null   object
 6   location          6735 non-null   object
 7   transaction_date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [43]:
# change data types
cols=["total_spent" , "price_per_unit"]
df[cols] = df[cols].apply(pd.to_numeric, errors = "coerce")

In [44]:
df["quantity"] = pd.to_numeric(df["quantity"] , errors = "coerce").astype("Int64")

In [45]:
df["transaction_date"]= pd.to_datetime(df["transaction_date"],format= "%d/%m/%Y", errors="coerce")

In [46]:
# checking missing values
df["item"].unique()[:20]

array(['Coffee', 'Cake', 'Cookie', 'Salad', 'Smoothie', 'UNKNOWN',
       'Sandwich', nan, 'ERROR', 'Juice', 'Tea'], dtype=object)

In [47]:
#dropping ERROR,UNKNOWN,NAN in "item"
bad_values =["ERROR","UNKNOWN" , ""]
df = df[~df["item"].isin(bad_values) & df["item"].notna()]

In [48]:
df["item"].unique()[:20]

array(['Coffee', 'Cake', 'Cookie', 'Salad', 'Smoothie', 'Sandwich',
       'Juice', 'Tea'], dtype=object)

In [49]:
# fill missing 'quantity'
mask = df["quantity"].isna() & df["price_per_unit"].notnull() & df["total_spent"].notnull()
df.loc[mask, "quantity"]= (df["total_spent"] / df["price_per_unit"])

In [50]:
df["quantity"]=df["quantity"].fillna(1)

In [51]:
# fill missing 'price_per_unit'
df["price_per_unit"] = df["price_per_unit"].fillna(df.groupby('item')["price_per_unit"].transform("median"))

In [52]:
# fill missing 'total_spent'
mask = df["total_spent"].isna()
df.loc[mask , "total_spent"] = (df.loc[mask , "quantity"] * df.loc[mask ,"price_per_unit"])

In [61]:
cols = ["payment_method" , "location"]
bad_values = ["UNKNOWN" , "ERROR" , ""]
df[cols]= (df[cols].replace(bad_values , "Unknown"). fillna("Unknown"))

In [62]:
df["payment_method"].unique()

array(['Credit Card', 'Cash', 'Unknown', 'Digital Wallet'], dtype=object)

In [54]:
bad_values = ["UNKOWN","ERRORS", ""]
df= df[~df["transaction_date"].isin(bad_values) & df["transaction_date"].notna()]

In [55]:
# calculating outliers

numeric_cols = ['quantity', 'price_per_unit', 'total_spent']

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    
    print(f"{col} Outliers: {len(outliers)}")

quantity Outliers: 0
price_per_unit Outliers: 0
total_spent Outliers: 239


In [56]:
from scipy import stats
import numpy as np

numeric_cols = ['quantity', 'price_per_unit', 'total_spent']

for col in numeric_cols:
    z_scores = np.abs(stats.zscore(df[col]))
    outliers = df[z_scores > 3]
    
    print(f"{col} Outliers: {len(outliers)}")

quantity Outliers: 0
price_per_unit Outliers: 0
total_spent Outliers: 0


In [63]:
df.to_csv(r"C:\Users\user\Desktop\Study sess\clean_cafe_sales.csv")